<div class="alert alert-block alert-info">

### Imports & Connection
</div>

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import datetime as datetime
import plotly.graph_objects as go

from scipy import stats
from pyhive import presto
from IPython.display import display
from datetime import datetime, timedelta
# from scipy.stats import gaussian_kde

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        # 'presto-gateway.processing.data.production.internal',
        # 'presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

<div class="alert alert-block alert-info">

### Datasets
</div>

In [5]:
def func_get_data():
    
    base_query = f"""
    
    select * from reports_internal.monetcluster_vivo_videobanner_26nov2024_view       
    union
    select * from reports_internal.monet_vivo_videobanner_21nov2024v4_view
    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

df = func_get_data()
df.shape

(12165522, 18)

In [6]:
df.to_parquet('/Users/E2074/local-datasets/customer/ads-monetisation/vivo-video-campaign/view.parquet', index=False)

In [7]:
df.head(2)

,yyyymmdd,userid,pagename,slotname,orderstatus,format,service,metadata1,response_id,impression_id,viewable_impression_id,click_id,taxi_ltr_segment,taxi_retention_segments,taxi_income_segment,taxi_service_affinity,taxi_regularity_segment,taxi_fe_regularity_segment
0,20241126,5f9533625acac61394184aaf,PostOrderScreen,PostOrderOnTheWay3,onTheWay,gamCombinedBanner,Link,"{title=vivo Y300, body=The style game just got...",0794493e-8277-4a54-a98a-6493815f70075f9533625a...,0794493e-8277-4a54-a98a-6493815f70075f9533625a...,0794493e-8277-4a54-a98a-6493815f70075f9533625a...,None,PHH,PLATINUM,LOW_INCOME,ONLY_LINK,BI_WEEKLY,BI_WEEKLY
1,20241126,6116027bae291c6733d5a67c,CaptainSearchScreen,CaptainSearch1,,gamCombinedBanner,Link,"{title=vivo Y300, body=The style game just got...",2b9e29ca-7447-4736-98c6-08158604a3f76116027bae...,2b9e29ca-7447-4736-98c6-08158604a3f76116027bae...,2b9e29ca-7447-4736-98c6-08158604a3f76116027bae...,None,PHH,ELITE,MEDIUM_INCOME,ONLY_LINK,DAILY,DAILY


<div class="alert alert-block alert-info">

### Data Manipulation
</div>

In [8]:
df['slot'] = df['slotname'].str.replace(r'\d+$', '', regex=True)
df['overall'] = 'Overall'

conditions = [
    df['yyyymmdd'] == '20241121',
    df['yyyymmdd'].isin(['20241122', '20241123', '20241124', '20241125']),
    df['yyyymmdd'] == '20241126'
]
choices = ['Learn More', 'Prebook Now', 'Buy Now']

df['campaign'] = np.select(conditions, choices, default='None')

In [9]:
df.shape

(12165522, 20)

<div class="alert alert-block alert-info">

### Analysis
</div>

In [10]:
dict_agg = {
    'response' : ('response_id', 'nunique'),
    'impression' : ('impression_id', 'nunique'),
    'viewable_impression' : ('viewable_impression_id', 'nunique'),
    'click' : ('click_id', 'nunique')
}

In [11]:
def func_get_funnel(df, groupby, dict_agg):
    
    temp_df = df\
                .groupby(groupby)\
                .agg(**dict_agg)\
                .reset_index()

    # temp_df['render rate'] = (temp_df['impression']*100.00/temp_df['response']).round(2)
    temp_df['viewability'] = (temp_df['viewable_impression']*100.00/temp_df['impression']).round(2)
    temp_df['ctr'] = (temp_df['click']*100.00/temp_df['viewable_impression']).round(2)
    
    return temp_df

In [12]:
func_get_funnel(df, ['overall'], dict_agg)

,overall,response,impression,viewable_impression,click,viewability,ctr
0,Overall,12164583,7110767,5827480,30002,81.95,0.51


In [13]:
func_get_funnel(df, ['pagename'], dict_agg)

,pagename,response,impression,viewable_impression,click,viewability,ctr
0,CaptainSearchScreen,3892090,2615074,2042314,14690,78.1,0.72
1,PostOrderScreen,8272493,4495693,3785166,15312,84.2,0.40


In [14]:
func_get_funnel(df, ['slot'], dict_agg)

,slot,response,impression,viewable_impression,click,viewability,ctr
0,CaptainSearch,3892090,2615074,2042314,14690,78.10,0.72
1,PostOrderArrived,2126008,704633,609710,1080,86.53,0.18
2,PostOrderOnTheWay,4055434,2629106,2175584,6493,82.75,0.30
3,PostOrderStarted,2091051,1161954,999872,7739,86.05,0.77


In [15]:
func_get_funnel(df, ['service'], dict_agg)

,service,response,impression,viewable_impression,click,viewability,ctr
0,,34544,23864,18655,92,78.17,0.49
1,Auto,4407068,2685751,2153141,9242,80.17,0.43
2,Auto C2C,6431,3261,2459,21,75.41,0.85
3,Auto NCR,15070,8738,7235,61,82.80,0.84
4,Auto Pet,1455,751,555,2,73.90,0.36
5,Auto Pool,16710,8000,5662,97,70.78,1.71
6,AutoPremium,833,496,378,2,76.21,0.53
7,Bike,3,2,2,0,100.00,0.00
8,Bike Lite,1212641,662764,557157,2668,84.07,0.48
9,Bike Metro,199348,112109,95965,346,85.60,0.36


In [16]:
func_get_funnel(df, ['taxi_ltr_segment'], dict_agg)

,taxi_ltr_segment,response,impression,viewable_impression,click,viewability,ctr
0,HH,1092534,669454,560190,3917,83.68,0.70
1,PHH,10821486,6317095,5171869,25091,81.87,0.49


In [17]:
func_get_funnel(df, ['taxi_retention_segments'], dict_agg)

,taxi_retention_segments,response,impression,viewable_impression,click,viewability,ctr
0,DORMANT,218711,104686,81559,821,77.91,1.01
1,ELITE,4274858,2439668,1966229,8571,80.59,0.44
2,GOLD,2183913,1312864,1090948,5758,83.10,0.53
3,HH,994987,623309,524466,3470,84.14,0.66
4,INACTIVE,87607,42033,33107,366,78.76,1.11
5,PLATINUM,2883731,1694015,1388846,6645,81.99,0.48
6,PRIME,188747,107820,88613,379,82.19,0.43
7,SILVER,1081466,662154,558291,2998,84.31,0.54


In [18]:
func_get_funnel(df, ['taxi_income_segment'], dict_agg)

,taxi_income_segment,response,impression,viewable_impression,click,viewability,ctr
0,HIGH_INCOME,5223721,3236117,2618989,12001,80.93,0.46
1,LOW_INCOME,772319,398743,331677,2279,83.18,0.69
2,MEDIUM_INCOME,4534130,2461621,2049254,10858,83.25,0.53
3,UNKNOWN,1383850,890068,732139,3870,82.26,0.53


In [19]:
func_get_funnel(df, ['taxi_regularity_segment'], dict_agg)

,taxi_regularity_segment,response,impression,viewable_impression,click,viewability,ctr
0,BI_WEEKLY,2368177,1404999,1164687,6123,82.90,0.53
1,DAILY,3744575,2122892,1709354,7548,80.52,0.44
2,MONTHLY,1014623,618994,516912,2820,83.51,0.55
3,RARE_NEED,325814,202442,170797,1015,84.37,0.59
4,UNKNOWN,1092534,669454,560190,3917,83.68,0.70
5,WEEKLY,3368297,1967768,1610119,7585,81.82,0.47
